# Grouped Planet Edge Model — Edge-Aware İşleyiş

Bu notebook `model_grouped_planet_edge.py` içindeki **EdgeAwareGroupedPlanetOrbitWarsPolicy** modelinin nasıl çalıştığını adım adım anlatır.

Amaç: eski `grouped_planet` modelini bozmadan, yeni `grouped_planet_edge` varyantında planet-planet edge feature'larını sadece attention score'a `+ bias` olarak değil, **mesaj içeriğine ve final action head'lerine de** doğrudan yedirmek.

Mimari özet:

1. `PlanetProjector` — her planet feature vektörünü tek `d_model` token'a çevirir.
2. `EdgeAwareMultiHeadAttention` — attention skoruna `edge_score(edge[src,tgt])` ekler.
3. Aynı attention içinde value mesajına `edge_value(edge[src,tgt])` eklenir.
4. PMA-style pooling — planet token'lardan global summary üretir.
5. Pair action heads:
   - `src_head` source launch skoru üretir.
   - `tgt_query/tgt_key` pairwise source-target skorunu üretir.
   - `pair_edge_score(edge[src,tgt])` final target logit'e eklenir.
   - `ship_head([src_token, tgt_token, edge_token, global_summary])` her pair için Beta parametresi üretir.
6. Rollout/watch tarafında `grouped_planet_edge`, `grouped_planet` ile aynı pair-ranking action path'ini kullanır.

> Kritik ayrım: `grouped_planet_edge` ayrı dosyada ve ayrı `model_variant` olarak yaşar. Eski `grouped_planet` sınıfı değiştirilmez.

In [1]:
import os, sys, pathlib
repo_root = pathlib.Path().resolve().parent if pathlib.Path.cwd().name == 'docs' else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)
print('repo_root =', repo_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)

repo_root = /home/doruk/Documents/Kaggle/Orbit_Wars
torch 2.11.0+cu128


## 1. Observation ve edge feature tensor'ları

Modelin kullandığı ana observation alanları:

| Alan | Shape | Anlam |
|---|---:|---|
| `planet_feats` | `(P, PLANET_FEATURE_DIM)` | planet başına owner/economy/geometry/dynamics/incoming/value feature'ları |
| `global_feats` | `(GLOBAL_FEATURE_DIM,)` | oyun geneli özet feature'lar |
| `planet_mask` | `(P,)` | aktif/padding planet maskesi |
| `source_mask` | `(P,)` | launch yapabilecek kendi planet slotları |
| `edge_bias` | `(P+F+1, P+F+1, EDGE_FEATURE_DIM)` | pairwise relation feature'ları |

Edge-aware model encoder tarafında yalnızca planet-planet bloğunu kullanır:

```python
edge_pair = edge_bias[:, :p_max, :p_max, :]
```

Yani attention ve final pair head'ler için edge tensor shape'i: `(B, P, P, EDGE_FEATURE_DIM)`.

In [4]:
from kaggle_environments import make
from obs_builder import build_observation, PLANET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM, P_MAX_DEFAULT
from model import pack_observations

env = make('orbit_wars', debug=False)
env.configuration['randomSeed'] = 42
state = env.reset(num_agents=2)
state = env.step([[], []])
raw_obs = state[0]['observation']
omega = raw_obs.get('angular_velocity', 0.0) or 0.0
obs = build_observation(raw_obs, player=0, angular_velocity=omega)
batch = pack_observations([obs], device=torch.device('cpu'))

print('P_MAX              =', P_MAX_DEFAULT)
print('PLANET_FEATURE_DIM =', PLANET_FEATURE_DIM)
print('GLOBAL_FEATURE_DIM =', GLOBAL_FEATURE_DIM)
print('EDGE_FEATURE_DIM   =', EDGE_FEATURE_DIM)
print()
for k in ['planet_feats', 'global_feats', 'edge_bias', 'planet_mask', 'source_mask']:
    v = batch[k]
    print(f'{k:13s} -> {tuple(v.shape)}')

edge_pair = batch['edge_bias'][:, :P_MAX_DEFAULT, :P_MAX_DEFAULT, :]
print('edge_pair     ->', tuple(edge_pair.shape), '  (B, P, P, edge_dim)')

P_MAX              = 44
PLANET_FEATURE_DIM = 34
GLOBAL_FEATURE_DIM = 14
EDGE_FEATURE_DIM   = 6

planet_feats  -> (1, 44, 34)
global_feats  -> (1, 14)
edge_bias     -> (1, 109, 109, 6)
planet_mask   -> (1, 44)
source_mask   -> (1, 44)
edge_pair     -> (1, 44, 44, 6)   (B, P, P, edge_dim)


In [5]:
obs

Observation(planet_feats=array([[0.     , 0.     , 0.     , ..., 0.     , 0.01996, 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.01996, 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.01996, 0.     ],
       ...,
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ],
       [0.     , 0.     , 0.     , ..., 0.     , 0.     , 0.     ]],
      shape=(44, 34), dtype=float32), planet_mask=array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False]), fleet_feats=array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0

## 2. Model varyantı ve eski modelden ayrım

Yeni model sınıfı:

```python
EdgeAwareGroupedPlanetOrbitWarsPolicy
```

Variant adı:

```python
model_variant = 'grouped_planet_edge'
```

Eski grouped model:

```python
GroupedPlanetOrbitWarsPolicy
model_variant = 'grouped_planet'
```

Bu ayrım checkpoint/factory/CLI tarafında önemlidir. `grouped_planet_edge` checkpoint'i eski `GroupedPlanetOrbitWarsPolicy` sınıfına yüklenmemelidir; `rl/train.py` factory bunu ayrı kurar.

In [6]:
from model_grouped_planet import GroupedPlanetOrbitWarsPolicy
from model_grouped_planet_edge import EdgeAwareGroupedPlanetOrbitWarsPolicy

D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2

old_policy = GroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX_DEFAULT,
    planet_dim=PLANET_FEATURE_DIM,
    global_dim=GLOBAL_FEATURE_DIM,
    edge_dim=EDGE_FEATURE_DIM,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    attention_type='full',
)
edge_policy = EdgeAwareGroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX_DEFAULT,
    planet_dim=PLANET_FEATURE_DIM,
    global_dim=GLOBAL_FEATURE_DIM,
    edge_dim=EDGE_FEATURE_DIM,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    attention_type='full',
)
edge_policy.eval()

for name, policy in [('old', old_policy), ('edge', edge_policy)]:
    print(f'{name:4s} variant = {policy.model_variant:20s} class = {type(policy).__name__}')
    print(f'{name:4s} params  = {sum(p.numel() for p in policy.parameters()):,}')

old  variant = grouped_planet       class = GroupedPlanetOrbitWarsPolicy
old  params  = 191,718
edge variant = grouped_planet_edge  class = EdgeAwareGroupedPlanetOrbitWarsPolicy
edge params  = 201,539


## 3. Edge-aware attention: sadece `+ bias` değil, edge mesajı da var

Eski `grouped_planet` full attention mantığı kabaca şuydu:

```text
scores[i,j,h] = q[i,h] · k[j,h] / sqrt(d_head) + edge_bias_proj(edge[i,j])[h]
out[i] = Σ_j softmax(scores[i,j]) · value[j]
```

Yeni `grouped_planet_edge` şunu yapar:

```text
scores[i,j,h] = q[i,h] · k[j,h] / sqrt(d_head) + edge_score(edge[i,j])[h]
attn[i,j,h]  = softmax_j(scores[i,j,h])
content[i,h] = Σ_j attn[i,j,h] · value[j,h]
edge_msg[i,h] = Σ_j attn[i,j,h] · edge_value(edge[i,j])[h]
out[i] = out_proj(content[i] + edge_msg[i])
```

Yani edge feature iki yerde kullanılır:

- **Nereye bakacağım?** `edge_score` attention distribution'ı değiştirir.
- **Baktığım yerden ne mesaj alacağım?** `edge_value` attention value içeriğine edge-conditioned mesaj ekler.

In [ ]:
attn = edge_policy.encoder.blocks[0].attn
print(attn)
print()
print('qkv weight       ->', tuple(attn.qkv.weight.shape))
print('edge_score      ->', tuple(attn.edge_score.weight.shape), '  edge_dim -> n_heads')
print('edge_value      ->', tuple(attn.edge_value.weight.shape), '  edge_dim -> d_model')
print('out_proj weight  ->', tuple(attn.out_proj.weight.shape))

### 3.1 Attention içindeki shape akışı

Bir batch için:

| Tensor | Shape |
|---|---:|
| `x` | `(B, P, d_model)` |
| `q,k,v` | `(B, H, P, d_head)` |
| `scores` | `(B, H, P, P)` |
| `edge_scores` | `(B, H, P, P)` |
| `attn` | `(B, H, P, P)` |
| `content_out` | `(B, H, P, d_head)` |
| `edge_value(edge_pair)` | `(B, P, P, d_model)` → `(B, H, P, P, d_head)` |
| `edge_out` | `(B, H, P, d_head)` |
| final output | `(B, P, d_model)` |

Bu işlem full attention olduğu için maliyet `O(P² · d_model)`; edge value mesajı eklenince memory de artar.

In [ ]:
with torch.no_grad():
    x0 = edge_policy.encoder.planet_proj(batch['planet_feats'])
    kpm = ~batch['planet_mask']
    B, P, D = x0.shape
    H = edge_policy.encoder.n_heads
    Dh = D // H
    qkv = attn.qkv(x0).reshape(B, P, 3, H, Dh).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    scores = torch.matmul(q, k.transpose(-2, -1)) / (Dh ** 0.5)
    edge_scores = attn.edge_score(edge_pair).permute(0, 3, 1, 2).contiguous()
    edge_v = attn.edge_value(edge_pair).reshape(B, P, P, H, Dh).permute(0, 3, 1, 2, 4)

print('x0          ->', tuple(x0.shape))
print('q/k/v       ->', tuple(q.shape))
print('scores      ->', tuple(scores.shape))
print('edge_scores ->', tuple(edge_scores.shape))
print('edge_v      ->', tuple(edge_v.shape))

## 4. Encoder: planet tokens + PMA global summary

Encoder sadece planet token dizisini işler:

```text
planet_feats -> PlanetProjector -> x: (B, P, d_model)
x -> N adet EdgeAwareTransformerBlock -> planet_tokens
planet_tokens + pool_seeds -> MultiheadAttention pool -> global_summary
```

Padding planet token'ları finalde maskelenir:

```python
planet_tok = x * planet_mask[:, :, None]
```

In [ ]:
with torch.no_grad():
    enc_out = edge_policy.encoder.encode_planets(
        planet_feats=batch['planet_feats'],
        fleet_feats=batch['fleet_feats'],
        global_feats=batch['global_feats'],
        edge_bias=batch['edge_bias'],
        planet_mask=batch['planet_mask'],
        fleet_mask=batch['fleet_mask'],
    )

for k, v in enc_out.items():
    print(f'{k:14s} -> shape={tuple(v.shape)} mean={v.float().mean().item():+.3f} std={v.float().std().item():.3f}')

pad_abs = enc_out['planet_tokens'][0, ~batch['planet_mask'][0]].abs().max().item()
print('padding token max abs ->', pad_abs)

## 5. Forward pass: final policy heads

Edge-aware policy head akışı:

```text
planet_tok, global_summary = encoder(...)
global_summary = summary_head([encoder_summary, global_feat_proj(global_feats)])

source_launch[s] = src_head([planet_tok[s], global_summary])
noop_raw[s]      = noop_head([planet_tok[s], global_summary])
src_logits       = [noop_raw[0:P], max(noop_raw)]

edge_tok[s,t]    = pair_edge_proj(edge[s,t])
q_s              = tgt_query(planet_tok[s])
k_t              = tgt_key(planet_tok[t])
tgt_logits[s,t]  = q_s · k_t / sqrt(d_model)
                 + source_launch[s]
                 + pair_edge_score(edge_tok[s,t])

ship_params[s,t] = ship_head([planet_tok[s], planet_tok[t], edge_tok[s,t], global_summary])
value            = value_head(global_summary)
aux_owner_logits = aux_owner_head(planet_tok)
```

Buradaki ana fark: final action score ve ship amount head doğrudan `edge[src,tgt]` görüyor.

In [ ]:
with torch.no_grad():
    out = edge_policy(**batch)

print('src_logits       ->', tuple(out.src_logits.shape), '  (B, P+1)')
print('tgt_logits       ->', tuple(out.tgt_logits.shape), '  (B, P, P)')
print('ship_params      ->', tuple(out.ship_params.shape), '  (B, P, P, 2)')
print('value            ->', tuple(out.value.shape))
print('tokens           ->', tuple(out.tokens.shape))
print('aux_owner_logits ->', tuple(out.aux_owner_logits.shape))

assert edge_policy.model_variant == 'grouped_planet_edge'
assert out.tgt_logits.shape[-2:] == (P_MAX_DEFAULT, P_MAX_DEFAULT)
assert out.ship_params.shape[-3:] == (P_MAX_DEFAULT, P_MAX_DEFAULT, 2)

### 5.1 Pair edge head'in final logit'e etkisi

`pair_edge_score(edge_tok[s,t])` final `tgt_logits[s,t]` üzerine eklenir.

Bu, action selection sırasında modelin doğrudan şunlara tepki verebilmesini sağlar:

- kaynak-hedef arası mesafe/relative geometry,
- hareketli hedef intercept zorluğu,
- trajectory/path relation feature'ları,
- source-target ilişkisine özel diğer edge feature'lar.

Eski `grouped_planet` modelinde edge bilgisi attention içinde contextual token üretimine dolaylı giriyordu. Yeni varyantta edge bilgisi action skorunun kendisine de direkt giriyor.

In [ ]:
with torch.no_grad():
    planet_tok = enc_out['planet_tokens']
    edge_tok = edge_policy.pair_edge_proj(edge_pair)
    edge_score_pair = edge_policy.pair_edge_score(edge_tok).squeeze(-1)

print('edge_tok         ->', tuple(edge_tok.shape))
print('edge_score_pair ->', tuple(edge_score_pair.shape))
print('edge score min/mean/max:',
      round(edge_score_pair.min().item(), 3),
      round(edge_score_pair.mean().item(), 3),
      round(edge_score_pair.max().item(), 3))

### 5.2 Ship head: pair-conditioned Beta dağılımı

Her `(src,tgt)` çifti için model iki raw değer üretir. Bunlar `_beta_concentrations` ile pozitif Beta concentration değerlerine çevrilir:

```text
raw_ship_params[s,t] -> alpha[s,t], beta[s,t]
ship_value ~ Beta(alpha[s,t], beta[s,t])
requested_ships ≈ ship_value × SHIP_REQUEST_CAP
```

`grouped_planet_edge` ship head input'u eski grouped modele göre daha zengindir:

```text
[src_token, tgt_token, edge_token, global_summary]
```

Bu yüzden ship amount kararı da source-target relation feature'ını doğrudan görür.

In [ ]:
from model import _beta_concentrations, _make_ship_dist
from obs_builder import SHIP_REQUEST_CAP

src_slots = [int(i) for i, m in enumerate(obs.source_mask) if m]
print('source slots:', src_slots)

if src_slots:
    s = src_slots[0]
    top_targets = torch.topk(out.tgt_logits[0, s], k=5)
    print(f'source_slot={s} için en yüksek 5 target ve Beta head:')
    for val, t in zip(top_targets.values.tolist(), top_targets.indices.tolist()):
        raw = out.ship_params[0, s, t]
        alpha, beta = _beta_concentrations(raw)
        dist = _make_ship_dist(raw)
        mean = (alpha / (alpha + beta)).item()
        sample = dist.sample().item()
        print(f'  tgt={t:3d} logit={val:+.3f}  alpha={alpha.item():.2f} beta={beta.item():.2f} '
              f'mean={mean:.3f} sample={sample:.3f} ships≈{sample * SHIP_REQUEST_CAP:.0f}')

## 6. Action mapping

Grouped/edge grouped policy'nin aksiyonu:

```text
NOOP
veya
source planet slot, target planet slot, ship_value
```

Joint action categorical mapping:

```text
action_idx = 0                  -> global NOOP
action_idx = src * P + tgt + 1  -> launch src -> tgt
```

Tersi:

```text
pair_idx = action_idx - 1
src = pair_idx // P
tgt = pair_idx % P
```

Rollout/watch ise bir turn içinde birden fazla launch denediği için pratikte source-target pair ranking yapar ve `evaluate_source_actions` ile log-prob hesaplar.

In [ ]:
P = edge_policy.p_max
examples = [(0, 1), (3, 10), (P - 1, P - 2)]
for s, t in examples:
    action_idx = s * P + t + 1
    pair_idx = action_idx - 1
    s_back = pair_idx // P
    t_back = pair_idx % P
    print(f'src={s:2d}, tgt={t:2d} -> action_idx={action_idx:4d} -> decoded=({s_back}, {t_back})')
print('total discrete alternatives =', 1 + P * P)

## 7. `act_source` ve `evaluate_source_actions`

Rollout içinde her source için conditional action kurulur:

```text
action_logits = [noop_raw[src], tgt_logits[src,0], tgt_logits[src,1], ..., tgt_logits[src,P-1]]
```

Eğer launch seçilirse:

```text
ship_value ~ Beta(alpha[src,tgt], beta[src,tgt])
log_prob = log p_categorical(tgt+1) + log p_beta(ship_value)
```

Eğer NOOP seçilirse:

```text
log_prob = log p_categorical(0)
```

Beta log-prob NOOP için eklenmez.

In [ ]:
if src_slots:
    s = src_slots[0]
    src_t = torch.tensor([s], dtype=torch.long)
    with torch.no_grad():
        action = edge_policy.act_source(out, source_slot=src_t, deterministic=True)
        ev = edge_policy.evaluate_source_actions(
            out,
            src=action['src'],
            tgt=action['tgt'],
            ship_value=action['ship_value'],
            is_noop=action['is_noop'],
        )
    print('action:')
    for k, v in action.items():
        print(f'  {k:10s} -> {v.tolist()}')
    print('recomputed log_prob:', ev['log_prob'].tolist())
    print('matches:', bool(torch.allclose(action['log_prob'], ev['log_prob'], atol=1e-5)))

## 8. Rollout/watch entegrasyonu

`rl/rollout.py` ve `rl/watch.py` içinde `grouped_planet_edge`, `grouped_planet` ile aynı grouped path'e girer:

```python
if model_variant in ('grouped_planet', 'grouped_planet_edge'):
    ... pair-ranking path ...
```

Turn içi seçim mantığı:

1. Observation modelden geçirilir.
2. `tgt_logits[0]` üzerinden tüm `(src,tgt)` pair'leri skorlanır.
3. Invalid pair'ler maskelenir: padding target, invalid source, `src == tgt`, aynı turn içinde denenmiş pair.
4. En iyi `MAX_PAIR_CANDIDATES` pair denenir.
5. Training rollout stochastic Beta sample kullanır.
6. Watch deterministic Beta mean kullanır.
7. Decode başarılıysa launch yapılır ve same-turn virtual observation güncellenir.
8. Bir player turn'ünde en fazla `MAX_LAUNCHES_PER_PLAYER_TURN` launch yapılır.

In [ ]:
from rl.watch import _act_all_sources, MAX_LAUNCHES_PER_PLAYER_TURN, MAX_PAIR_CANDIDATES

print('MAX_LAUNCHES_PER_PLAYER_TURN =', MAX_LAUNCHES_PER_PLAYER_TURN)
print('MAX_PAIR_CANDIDATES          =', MAX_PAIR_CANDIDATES)

actions, infos = _act_all_sources(edge_policy, raw_obs, player=0, deterministic=True, device=torch.device('cpu'))
print('env action count:', len(actions))
for info in infos[:8]:
    print(f'  src={info["src_slot"]:3d} tgt={info["tgt_slot"]:3d} '
          f'ship_value={info["ship_value"]:.3f} value={info["value"]:+.3f}')

## 9. Checkpoint/CLI entegrasyonu

Training CLI tarafında model şöyle seçilir:

```bash
orbit_wars_venv/bin/python -m rl.train \
  --model-variant grouped_planet_edge \
  --grouped-attention-type full \
  --reward-system production_simple
```

Checkpoint `arch` metadata içinde variant saklanır:

```text
arch['model_variant'] = 'grouped_planet_edge'
```

Load ederken factory bu variant'a göre `EdgeAwareGroupedPlanetOrbitWarsPolicy` kurar. Böylece eski `grouped_planet` checkpoint'leriyle yeni edge checkpoint'leri karışmaz.

Dikkat: `grouped_planet_edge` şu an sadece `attention_type='full'` destekler. `induced` verilirse hata verir.

In [ ]:
try:
    EdgeAwareGroupedPlanetOrbitWarsPolicy(
        p_max=P_MAX_DEFAULT,
        planet_dim=PLANET_FEATURE_DIM,
        global_dim=GLOBAL_FEATURE_DIM,
        edge_dim=EDGE_FEATURE_DIM,
        d_model=32,
        n_heads=4,
        n_layers=1,
        attention_type='induced',
    )
except ValueError as e:
    print('expected error:', e)

## 10. Bellek ve training stabilitesi notları

Edge-aware model eski grouped modele göre daha pahalıdır. En pahalı yerlerden biri ship head input'udur:

```python
src_pair    = planet_tok[:, :, None, :]       # (B, P, 1, D) -> expand (B, P, P, D)
tgt_pair    = planet_tok[:, None, :, :]       # (B, 1, P, D) -> expand (B, P, P, D)
edge_tok    = pair_edge_proj(edge_pair)       # (B, P, P, D)
global_pair = global_summary[:, None, None, :]# (B, 1, 1, D) -> expand (B, P, P, D)
ship_input  = cat([...], dim=-1)              # (B, P, P, 4D)
```

Bu yüzden PPO batch size büyüdükçe VRAM hızlı artar. `P=44`, `D=128` ve büyük batch ile OOM görülebilir.

Training sırasında takip edilecek metrikler:

- `approx_kl`: target'ın çok üstüne çıkmamalı.
- `clip_fraction`: sürekli `0.4+` ise update agresif.
- `ratio_q95`: `3+` riskli, çok büyük değerler divergence işareti.
- `entropy`: çok hızlı çöküyorsa policy erken kilitleniyor.
- `P0/P1/D`: scripted'e karşı uzun süre P0=0 kalıyorsa strategy/reward/rollout incelenmeli.

PPO tarafında minibatch bazlı KL/finite guard kullanılır; divergence olmuş checkpoint'ten resume edilmemelidir.

## 11. Eski `grouped_planet` ile farkların kısa özeti

| Konu | `grouped_planet` | `grouped_planet_edge` |
|---|---|---|
| Dosya | `model_grouped_planet.py` | `model_grouped_planet_edge.py` |
| Variant | `grouped_planet` | `grouped_planet_edge` |
| Attention edge kullanımı | score bias | score bias + edge-conditioned value message |
| Final target logit | `q·k + source_launch` | `q·k + source_launch + pair_edge_score(edge)` |
| Ship head input | `[src_token, tgt_token, global]` tarzı pair context | `[src_token, tgt_token, edge_token, global]` |
| Attention type | `full` veya `induced` | sadece `full` |
| Rollout path | grouped pair-ranking | aynı grouped pair-ranking path |
| Checkpoint load | kendi sınıfı | kendi sınıfı |

En önemli fikir: edge feature artık yalnızca token context üretimine dolaylı etki etmiyor; **hangi hedefe saldırılacağı** ve **ne kadar gemi gönderileceği** kararlarının içine doğrudan giriyor.